# 04 — Results & Outputs
**Wildfire Risk Modeling Exercise | Cell2Fire W — Scott & Burgan**

Reads simulation grids and produces all submission deliverables:

| Output | File | Destination |
|---|---|---|
| Burn extent raster | `burn_extent.tif` | Wildfire Commons + AGOL |
| Fire perimeter | `fire_perimeter.geojson` | AGOL Feature Layer |
| Building exposure | `building_exposure.geojson` | AGOL Feature Layer |
| Results figure | `02_simulation_results.png` | Report |
| Stats summary | printed | Report |

In [ ]:
TOWN = "forest"   # "forest" or "prairie"

In [ ]:
# ============================================================
# CELL 1 — Imports, config, paths
# ============================================================
import sys, pathlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import shapes
from shapely.geometry import shape, mapping, MultiPoint
from shapely.ops import unary_union

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils import load_config

cfg      = load_config(TOWN, REPO_ROOT)
INSTANCE = REPO_ROOT / cfg["instance_dir"]
RESULTS  = REPO_ROOT / cfg["results_dir"]
PLOTS    = REPO_ROOT / cfg["plots_dir"]
RAW      = REPO_ROOT / cfg["data_raw"]
PLOTS.mkdir(parents=True, exist_ok=True)

f        = cfg["files"]
sim_cfg  = cfg["simulation"]
BLDG_DIR = RAW / f"{TOWN}-building-data"

# Read ASC header for grid geometry
def read_asc_header(path):
    hdr = {}
    with open(path) as fh:
        for _ in range(6):
            k, v = fh.readline().split()
            hdr[k.lower()] = float(v)
    return hdr

hdr      = read_asc_header(INSTANCE / "fuels.asc")
ncols    = int(hdr["ncols"])
nrows    = int(hdr["nrows"])
cellsize = hdr["cellsize"]
left     = hdr["xllcorner"]
bottom   = hdr["yllcorner"]
right    = left + cellsize * ncols
top      = bottom + cellsize * nrows
extent   = [left, right, bottom, top]

fuels_data = np.loadtxt(INSTANCE / "fuels.asc", skiprows=6)

# Ignition location from Ignitions.csv
ign_df   = pd.read_csv(INSTANCE / "Ignitions.csv")
cell_num = int(ign_df["Ncell"].iloc[0])
ign_row  = (cell_num - 1) // ncols
ign_col  = (cell_num - 1) %  ncols
IGN_X    = left + (ign_col + 0.5) * cellsize
IGN_Y    = bottom + (nrows - ign_row - 0.5) * cellsize

print(f"Town:     {cfg['display_name']}")
print(f"Grid:     {nrows}×{ncols}  |  cellsize={cellsize:.1f}m")
print(f"Extent:   {left:.0f}–{right:.0f} E, {bottom:.0f}–{top:.0f} N")
print(f"Ignition: row={ign_row}, col={ign_col} → X={IGN_X:.0f}, Y={IGN_Y:.0f}")
print("✓ Cell 1 ready")

In [ ]:
# ============================================================
# CELL 2 — Load grid timesteps + compute results
# ============================================================
grid_dir   = RESULTS / "Grids" / "Grids1"
grid_files = sorted(
    grid_dir.glob("*.csv"),
    key=lambda x: int(''.join(filter(str.isdigit, x.stem)))
)[1:]  # skip Grid0 (empty initial state)

n_steps = len(grid_files)
print(f"Grid timesteps: {n_steps}")

ignition_time   = np.full((nrows, ncols), np.nan)
burned_per_step = []
prev            = np.zeros((nrows, ncols))

for i, gf in enumerate(grid_files):
    grid         = np.genfromtxt(gf, delimiter=",")
    newly_burned = (grid == 1) & (prev == 0)
    ignition_time[newly_burned] = i + 1
    burned_per_step.append(int((grid == 1).sum()))
    prev = grid

final_grid     = prev
total_burned   = burned_per_step[-1]
burned_area_ha = total_burned * (cellsize ** 2) / 10000
hours          = np.linspace(0, sim_cfg["duration_hours"], n_steps)
area_ha_steps  = [b * (cellsize**2) / 10000 for b in burned_per_step]

print(f"Total burned:    {total_burned:,} cells = {burned_area_ha:.1f} ha")
print(f"Non-burnable:    {(fuels_data == 0).sum() / fuels_data.size * 100:.1f}%")

# Buildings reprojected to EPSG:5070
bldgs = gpd.read_file(BLDG_DIR / f["buildings"]).to_crs(cfg["crs_out"])

# Exposure: buildings within convex hull of burned area
burned_rows, burned_cols = np.where(~np.isnan(ignition_time))
burned_xs = left   + (burned_cols + 0.5) * cellsize
burned_ys = bottom + (nrows - burned_rows - 0.5) * cellsize
burn_poly = MultiPoint(list(zip(burned_xs, burned_ys))).convex_hull
bldgs["exposed"] = bldgs.geometry.intersects(burn_poly)
n_exposed = int(bldgs["exposed"].sum())

fuels_plot = fuels_data.copy().astype(float)
fuels_plot[fuels_plot == 0] = np.nan

print(f"\nBuildings total: {len(bldgs):,}")
print(f"Exposed:         {n_exposed:,} ({n_exposed/len(bldgs)*100:.1f}%)")
print("✓ Cell 2 ready")

In [ ]:
# ============================================================
# CELL 3 — Export GeoTIFF, perimeter GeoJSON, building exposure GeoJSON
# ============================================================
from rasterio.crs import CRS

crs_out  = CRS.from_string(cfg["crs_out"])
transform = from_bounds(left, bottom, right, top, ncols, nrows)

# ── 1. Burn extent GeoTIFF ────────────────────────────────────
burn_tif = RESULTS / "burn_extent.tif"
burn_arr = final_grid.astype(np.uint8)
with rasterio.open(
    burn_tif, "w",
    driver="GTiff", crs=crs_out, transform=transform,
    width=ncols, height=nrows, count=1,
    dtype="uint8", nodata=255
) as dst:
    dst.write(burn_arr, 1)
print(f"✓ burn_extent.tif  ({burn_tif.stat().st_size/1024:.0f} KB)")

# ── 2. Fire perimeter GeoJSON ─────────────────────────────────
mask = (burn_arr == 1).astype(np.uint8)
polys = [
    shape(s) for s, v in
    shapes(mask, mask=(mask == 1), transform=transform)
    if v == 1
]
if polys:
    perimeter = unary_union(polys)
    perim_gdf = gpd.GeoDataFrame(
        [{"burned_area_ha": round(burned_area_ha, 2),
          "burned_cells":   total_burned,
          "sim_model":      "Cell2FireW_SB40",
          "nsims":          sim_cfg["nsims"]}],
        geometry=[perimeter], crs=cfg["crs_out"]
    )
    perim_path = RESULTS / "fire_perimeter.geojson"
    perim_gdf.to_crs("EPSG:4326").to_file(perim_path, driver="GeoJSON")
    print(f"✓ fire_perimeter.geojson  ({perim_path.stat().st_size/1024:.0f} KB)")

# ── 3. Building exposure GeoJSON ──────────────────────────────
bldgs_out = bldgs[["bldgID", "usage", "woodRoof", "woodSiding",
                   "exposed", "geometry"]].copy()
bldgs_out["exposed"] = bldgs_out["exposed"].astype(int)
exp_path = RESULTS / "building_exposure.geojson"
bldgs_out.to_crs("EPSG:4326").to_file(exp_path, driver="GeoJSON")
print(f"✓ building_exposure.geojson  ({exp_path.stat().st_size/1024:.0f} KB)")

print("\n✓ Cell 3 ready — all deliverable files exported")

In [ ]:
# ============================================================
# CELL 4 — Results visualisation (4-panel figure)
# ============================================================
cmap_fire = mcolors.LinearSegmentedColormap.from_list(
    "fire", ["#ffffb2","#fecc5c","#fd8d3c","#f03b20","#bd0026"]
)

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        width_ratios=[1, 1, 0.8],
                        hspace=0.35, wspace=0.3)

ax_final = fig.add_subplot(gs[:, 0])
ax_prog  = fig.add_subplot(gs[:, 1])
ax_area  = fig.add_subplot(gs[0, 2])
ax_stats = fig.add_subplot(gs[1, 2])

# ── Panel 1: Final burn map ───────────────────────────────────
ax = ax_final
ax.imshow(fuels_plot, extent=extent, origin="upper",
          cmap="YlGn", alpha=0.45, interpolation="nearest")
ax.imshow(np.where(fuels_data == 0, 1, np.nan), extent=extent,
          origin="upper", cmap="gray", alpha=0.2, interpolation="nearest")
ax.imshow(np.where(final_grid == 1, 1, np.nan), extent=extent,
          origin="upper", cmap="Reds", alpha=0.85, interpolation="nearest")
bldgs[~bldgs["exposed"]].plot(ax=ax, color="steelblue",
                               markersize=2, alpha=0.6, zorder=4)
bldgs[bldgs["exposed"]].plot(ax=ax, color="orange",
                              markersize=4, alpha=1.0, zorder=5)
ax.plot(IGN_X, IGN_Y, "*", color="yellow", markersize=16,
        markeredgecolor="black", markeredgewidth=1, zorder=6)
ax.legend(handles=[
    mpatches.Patch(facecolor="#d73027", label=f"Burned ({burned_area_ha:.1f} ha)"),
    mpatches.Patch(facecolor="#91cf60", alpha=0.6, label="Burnable fuel"),
    mpatches.Patch(facecolor="grey", alpha=0.4, label="Non-burnable"),
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="steelblue",
               markersize=6, label=f"Buildings ({len(bldgs):,})"),
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="orange",
               markersize=6, label=f"Exposed ({n_exposed:,})"),
    plt.Line2D([0],[0], marker="*", color="w", markerfacecolor="yellow",
               markeredgecolor="black", markersize=12, label="Ignition"),
], loc="upper left", fontsize=8, framealpha=0.9)
ax.set_title("Final Burn Map", fontsize=12, fontweight="bold")
ax.set_xlabel(f"Easting (m, {cfg['crs_out']})", fontsize=9)
ax.set_ylabel(f"Northing (m, {cfg['crs_out']})", fontsize=9)

# ── Panel 2: Fire progression ─────────────────────────────────
ax = ax_prog
ax.imshow(fuels_plot, extent=extent, origin="upper",
          cmap="Greys", alpha=0.25, interpolation="nearest")
im = ax.imshow(ignition_time, extent=extent, origin="upper",
               cmap=cmap_fire, alpha=0.9, interpolation="nearest",
               vmin=1, vmax=n_steps)
cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Fire arrival", fontsize=9)
ticks = np.linspace(1, n_steps, 5).astype(int)
cbar.set_ticks(ticks)
cbar.set_ticklabels([f"T+{h:.0f}h"
                     for h in np.linspace(0, sim_cfg['duration_hours'], 5)])
bldgs[bldgs["exposed"]].plot(ax=ax, color="orange",
                              markersize=4, alpha=1.0, zorder=5)
ax.plot(IGN_X, IGN_Y, "*", color="white", markersize=16,
        markeredgecolor="black", markeredgewidth=1, zorder=6)
ax.set_title("Fire Progression", fontsize=12, fontweight="bold")
ax.set_xlabel(f"Easting (m, {cfg['crs_out']})", fontsize=9)
ax.set_ylabel(f"Northing (m, {cfg['crs_out']})", fontsize=9)

# ── Panel 3: Burned area growth ───────────────────────────────
ax = ax_area
ax.plot(hours, area_ha_steps, color="#bd0026",
        linewidth=2.5, marker="o", markersize=5)
ax.fill_between(hours, area_ha_steps, alpha=0.15, color="#bd0026")
for i, (h, a) in enumerate(zip(hours, area_ha_steps)):
    if i % 3 == 0:
        ax.annotate(f"{a:.0f}ha", xy=(h, a),
                    xytext=(3, 5), textcoords="offset points", fontsize=7)
ax.set_xlabel("Time (hours)", fontsize=9)
ax.set_ylabel("Burned area (ha)", fontsize=9)
ax.set_title("Burned Area Growth", fontsize=10, fontweight="bold")
ax.set_xlim(0, sim_cfg["duration_hours"])
ax.set_ylim(0)
ax.grid(True, alpha=0.3)

# ── Panel 4: Summary table ────────────────────────────────────
ax = ax_stats
ax.axis("off")
stats = [
    ["Simulator",    "Cell2Fire W"],
    ["Fuel model",   "Scott & Burgan (S&B)"],
    ["Simulations",  str(sim_cfg["nsims"])],
    ["Grid",         f"{nrows}×{ncols} cells"],
    ["Cell size",    f"{cellsize:.1f} m"],
    ["Duration",     f"{sim_cfg['duration_hours']} hours"],
    ["Burned area",  f"{burned_area_ha:.1f} ha"],
    ["Burned cells", f"{total_burned:,}"],
    ["Buildings",    f"{len(bldgs):,} total"],
    ["Exposed",      f"{n_exposed:,} ({n_exposed/len(bldgs)*100:.1f}%)"],
]
tbl = ax_stats.table(cellText=stats, loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.5)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("lightgrey")
    if c == 0:
        cell.set_facecolor("#f0f0f0")
        cell.set_text_props(fontweight="bold")
ax_stats.set_title("Simulation Summary", fontsize=10, fontweight="bold")

fig.suptitle(
    f"Wildfire Risk Modeling Exercise — Cell2Fire W\n"
    f"UL Research Institutes | {cfg['display_name']} | Scott & Burgan",
    fontsize=13, fontweight="bold", y=1.01
)
out_fig = PLOTS / "02_simulation_results.png"
plt.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Cell 4 ready — figure saved to {out_fig}")

In [ ]:
# ============================================================
# CELL 5 — Deliverables checklist
# ============================================================
print("── Submission deliverables ──────────────────────────")
deliverables = {
    RESULTS / "burn_extent.tif":         "Burn extent raster (GeoTIFF)",
    RESULTS / "fire_perimeter.geojson":  "Fire perimeter (GeoJSON → AGOL)",
    RESULTS / "building_exposure.geojson": "Building exposure (GeoJSON → AGOL)",
    PLOTS   / "02_simulation_results.png": "Results figure (report)",
}
all_ok = True
for path, desc in deliverables.items():
    ok   = path.exists()
    size = path.stat().st_size / 1024 if ok else 0
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗'}  {path.name:<35} "
          f"{size:>7.0f} KB  {desc}")

print(f"\n{'✓ All deliverables ready — proceed to AGOL upload' if all_ok else '✗ Fix missing files above'}")